# tiny-rlhf: run every remaining experiment (Colab)

This notebook has **one cell**. Run only that cell; do not use Runtime > Run all in other notebooks.

**Before running, once per notebook:**
1. **Runtime > Change runtime type > T4 GPU**, then Save.
2. **Key icon on the left (Secrets):** a secret named `GH_TOKEN` holding your GitHub token, with **Notebook access** switched on.

**What it does:** it downloads the project, runs the tests, then works through every experiment that is not finished yet, in priority order, pushing each result to GitHub as soon as it finishes. If it stops for any reason (internet, Colab limits, closing the tab), run it again later: finished work is skipped and it continues where it left off.

**Healthy start:** within about 10 seconds you see `$ git clone ...`, then `tests passed` after about a minute, then the first experiment. For KTO the first training line shows `loss 0.5000`.

If it fails, send Claude the last lines of the output, or a screenshot.


In [ ]:
# tiny-rlhf: run the remaining GPU jobs on a Colab T4, pushing after each one.
BUDGET_MIN = 720  # 12 hours, Colab's maximum: in practice, run until done or cut off
STAGES = ""  # "" for everything, or for example "m3" or "beta,lora_rank"
BRANCH = "claude/charming-cray-20zxv9"
REPO = "Varkot-dev/tiny-rlhf"

import base64, os, shutil, subprocess, time
import torch
from google.colab import userdata

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
START = time.time()

# Give git the token through environment variables, so it is never written to disk or printed.
auth = base64.b64encode(f"x-access-token:{userdata.get('GH_TOKEN')}".encode()).decode()
os.environ.update(
    {
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
        "GIT_TERMINAL_PROMPT": "0",
        "PYTHONUNBUFFERED": "1",
    }
)


def run(cmd, show=True, check=True):
    """Run a shell command and print its output live. Returns the exit status."""
    print(f"\n$ {cmd}   [{(time.time() - START) / 60:.1f} min]", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if show and "Warning" not in line and "warnings.warn" not in line:
            print(line, end="", flush=True)
    code = proc.wait()
    if code != 0 and not show:
        print("".join(lines[-40:]))
    if code != 0 and check:
        raise RuntimeError(f"exit {code}: {cmd}")
    return code


os.chdir("/content")
shutil.rmtree("/content/tiny-rlhf", ignore_errors=True)
run(f"git clone -q --branch {BRANCH} https://github.com/{REPO}.git /content/tiny-rlhf")
os.chdir("/content/tiny-rlhf")
run('pip install -q -e ".[train,viz,dev]"')
run("pip uninstall -y -q torchao")  # Colab's old torchao breaks the LoRA library; unused here
run('git config user.name "Varshith Kotagiri"')
run('git config user.email "Varkot-dev@users.noreply.github.com"')
if run("python -m pytest -q -x", show=False, check=False) == 0:
    print("tests passed")
else:
    print("TESTS FAILED (output above). The queue still runs: each job has its own checks.")

left = BUDGET_MIN - (time.time() - START) / 60
stages = f"--stages {STAGES}" if STAGES else ""
code = run(
    f"python scripts/colab_queue.py --budget-min {left:.0f} --branch {BRANCH} {stages}",
    check=False,
)
print("\nQueue finished." if code == 0 else "\nQueue finished; some jobs failed (pushed).")
run("python scripts/colab_queue.py --list")